In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
liheap_validated = pd.read_csv("LIHEAP_validated.csv")
liheap_validated.head(5)

,Zip Code,Month Year,Enrollment_Count,Is_Negative,Outlier
0,91901,2023-02,1,False,False
1,91901,2023-03,2,False,False
2,91901,2023-04,1,False,False
3,91901,2023-05,2,False,False
4,91901,2023-06,2,False,False


**Define Lags Variable**

In [3]:
LAGS = (1, 2, 3)

In [4]:
liheap_validated.dtypes

Zip Code            int64
Month Year            str
Enrollment_Count    int64
Is_Negative          bool
Outlier              bool
dtype: object

In [5]:
liheap_validated = liheap_validated.rename(columns={"Month Year": "Month_Year"})

**Data Handling**
- zip codes as 5 digit strings
- convert 2023-02 into a real monthly period so that the lagging computations can commence

In [6]:
liheap_validated["Zip Code"] = liheap_validated["Zip Code"].astype(str).str.zfill(5)
liheap_validated["Month_Year"] = pd.PeriodIndex(liheap_validated["Month_Year"], freq="M")
liheap_validated = liheap_validated.sort_values(["Zip Code", "Month_Year"]).reset_index(drop=True)

In [7]:
liheap_validated

,Zip Code,Month_Year,Enrollment_Count,Is_Negative,Outlier
0,91901,2023-02,1,False,False
1,91901,2023-03,2,False,False
2,91901,2023-04,1,False,False
3,91901,2023-05,2,False,False
4,91901,2023-06,2,False,False
...,...,...,...,...,...
573,92177,2023-06,1,False,False
574,92193,2023-04,1,False,False
575,92195,2023-05,1,False,False
576,92198,2023-01,1,False,False


**Define Cell Lag Function**

In [8]:
def add_lags(df, lags=LAGS):
    """Add LIHEAP_lag{k} = enrollment in the same zip, k calendar months earlier."""
    base = df[["Zip Code", "Month_Year", "Enrollment_Count"]]
    out = df.copy()
    for k in lags:
        lagged = base.copy()
        lagged["Month_Year"] = lagged["Month_Year"] + k
        lagged = lagged.rename(columns={"Enrollment_Count": f"LIHEAP_lag{k}"})
        out = out.merge(lagged, on=["Zip Code", "Month_Year"], how="left")
    return out

**Define a Full Zip and month grid with lags**
- The zip code rows for august, september and octoboer 2023 do not have any enrollments because the data doesn't cover it
- However, because of the lags, they are populated to showcase the enrollment in previous months

In [9]:
#return every zip and month combination from janauary 2023 to july 2023 with a three month lasg (October 2023)
months = pd.period_range(liheap_validated["Month_Year"].min(),
                        liheap_validated["Month_Year"].max() + max(LAGS), freq="M")

grid = pd.MultiIndex.from_product(
    [liheap_validated["Zip Code"].unique(), months],
    names=["Zip Code", "Month_Year"]
).to_frame(index=False)

#return actual enrollment, months with nulls stay nan, they do not get turned to 0
grid = grid.merge(liheap_validated[["Zip Code", "Month_Year", "Enrollment_Count"]],
                 on=["Zip Code", "Month_Year"], how="left")

liheap_lags = add_lags(grid)

#keep columns with at least one lag value 
lag_cols = [f"LIHEAP_lag{k}" for k in LAGS]
liheap_lags = liheap_lags.dropna(subset=lag_cols, how="all").reset_index(drop=True)

liheap_lags.head(10)

,Zip Code,Month_Year,Enrollment_Count,LIHEAP_lag1,LIHEAP_lag2,LIHEAP_lag3
0,91901,2023-03,2.0,1.0,NaN,NaN
1,91901,2023-04,1.0,2.0,1.0,NaN
2,91901,2023-05,2.0,1.0,2.0,1.0
3,91901,2023-06,2.0,2.0,1.0,2.0
4,91901,2023-07,1.0,2.0,2.0,1.0
5,91901,2023-08,NaN,1.0,2.0,2.0
6,91901,2023-09,NaN,NaN,1.0,2.0
7,91901,2023-10,NaN,NaN,NaN,1.0
8,91902,2023-02,3.0,2.0,NaN,NaN
9,91902,2023-03,3.0,3.0,2.0,NaN


**Perform spot check on zip codes 91901, 91910, 91944, 92106, 92059**

In [10]:
def spot_check(df, lagged, zips, lags=LAGS):
    """Print raw series next to lagged rows, and verify every lag against the raw data."""
    raw = df.set_index(["Zip Code", "Month_Year"])["Enrollment_Count"]
    problems=0
    for z in zips:
        print(f"\n=== Zip {z} ===")
        print("Raw:", ", ".join(f"{m}={v}" for (zz, m), v in raw.loc[[z]].items()))
        sub = lagged[lagged["Zip Code"] == z]
        print(sub[["Month_Year", "Enrollment_Count"] + [f"LIHEAP_lag{k}" for k in lags]]
             .to_string(index=False))
        for _, row in sub.iterrows():
            for k in lags:
                expected = raw.get((z, row["Month_Year"] - k))
                got = row[f"LIHEAP_lag{k}"]
                if not ((pd.isna(got) and expected is None) or got == expected):
                    problems += 1
                    print(f"  MISMATCH {row['Month']} lag{k}: got {got}, expected {expected}")
    print(f"\nSpot-check done: {problems} mismatches") 

In [11]:
spot_check(liheap_validated, liheap_lags, ["91901", "91910", "91944", "92106", "92059"])


=== Zip 91901 ===
Raw: 2023-02=1, 2023-03=2, 2023-04=1, 2023-05=2, 2023-06=2, 2023-07=1
Month_Year  Enrollment_Count  LIHEAP_lag1  LIHEAP_lag2  LIHEAP_lag3
   2023-03               2.0          1.0          NaN          NaN
   2023-04               1.0          2.0          1.0          NaN
   2023-05               2.0          1.0          2.0          1.0
   2023-06               2.0          2.0          1.0          2.0
   2023-07               1.0          2.0          2.0          1.0
   2023-08               NaN          1.0          2.0          2.0
   2023-09               NaN          NaN          1.0          2.0
   2023-10               NaN          NaN          NaN          1.0

=== Zip 91910 ===
Raw: 2023-01=5, 2023-02=15, 2023-03=8, 2023-04=6, 2023-05=12, 2023-06=6, 2023-07=5
Month_Year  Enrollment_Count  LIHEAP_lag1  LIHEAP_lag2  LIHEAP_lag3
   2023-02              15.0          5.0          NaN          NaN
   2023-03               8.0         15.0          5.0       

In [12]:
liheap_lags.dtypes

Zip Code                  str
Month_Year          period[M]
Enrollment_Count      float64
LIHEAP_lag1           float64
LIHEAP_lag2           float64
LIHEAP_lag3           float64
dtype: object

**Output the LiHEAP Lags csv**

In [13]:
liheap_lags.assign(Month=liheap_lags["Month_Year"].astype(str)).to_csv("LIHEAP_lags.csv", index=False)
print(f"Saved LIHEAP_lags.csv: {len(liheap_lags)} rows")

Saved LIHEAP_lags.csv: 852 rows


**Handle Edge Cases Presented By The Data**

In [14]:
#define the start and end date of the data

In [15]:
data_start = liheap_validated["Month_Year"].min()
data_end = liheap_validated["Month_Year"].max()

In [16]:
# define why each lag is presnet or missing 
for k in LAGS:
    lag_col = f"LIHEAP_lag{k}"
    lag_month = liheap_lags["Month_Year"] - k
    liheap_lags[f"lag{k}_status"] = np.select(
        [liheap_lags[lag_col].notna(),
        lag_month < data_start,
        lag_month > data_end],
        ["present", "before_start", "after_end"],
        default="zip_gap"
    )

status_cols = [f"lag{k}_status" for k in LAGS]
status = liheap_lags[status_cols]

In [17]:
#Return the row level flags 
liheap_lags["Complete_Lags"] = status.eq("present").all(axis=1)
liheap_lags["Edge"] = np.select(
    [status.eq("before_start").any(axis=1),
    status.eq("after_end").any(axis=1)],
    ["start", "end"],
    default="core"
)

pd.crosstab(liheap_lags["Edge"], liheap_lags["Complete_Lags"])

Complete_Lags,False,True
Edge,,
core,168,346
end,165,0
start,173,0


**Conclusions**
- there are 346 rows where all three lags are present 
- There are 168 inside the January 2023 to July 2023 where at least one lag does not have a zip to record, these are gaps in the data for each zip code not edge effects, low enrollment zips
- There are 173 rows / 165 end rows that are missing lags purely because of where the data beings and ends. None of them can ever be complete, which is why the true column is 0 for both

**How Many Zip Codes Contribute to Complete Rows**
- 77 of 121 zip codes (64%) contribute at least one complete-lag row; the remaining 44 have sporadic LIHEAP records and contribute no complete rows. Contributing zips average 4–5 complete months (maximum possible: 5, Apr–Aug 2023). Complete-lag results therefore reflect higher-activity zip codes.

In [18]:
zip_complete = liheap_lags.groupby("Zip Code")["Complete_Lags"].sum()

print(f"Zips total: {zip_complete.size}")
print(f"Zips with at least 1 complete row: {(zip_complete > 0).sum()}")
zip_complete.describe()

Zips total: 121
Zips with at least 1 complete row: 77


count    121.000000
mean       2.859504
std        2.335615
min        0.000000
25%        0.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: Complete_Lags, dtype: float64

In [19]:
liheap_lags.assign(Month_Year=liheap_lags["Month_Year"].astype(str)).to_csv("LIHEAP_lags.csv", index=False)
print(f"Saved LIHEAP_lags.csv: {len(liheap_lags)} rows, {liheap_lags.shape[1]} columns")

Saved LIHEAP_lags.csv: 852 rows, 11 columns


**<p>Justification for Data Handling Decisions</p>**
- The incomplete rows were retained and labeled with the reason that each lag is missing. This analysis is consistent with the data validation step were missing months and outliers were retained rather than imputed or excluded
**<p>Start of Series</p>**
- The LIHEAP zip-level data begins in January 2023, so lag1 is unavailable in January, lag 2 for January-February, and lag3 for January-March. In the edge column, these rows are marked as "start". These values are kept as no earlier data exists, which contributes to the hole in the dates where zip codes did not contain enrollments
**<p>End of Series</p>**
  - LIHEAP zip-level data ends in July 2023, while CalFresh and FoodBank data extend beyond that. The outcome months extend to October 2023, with lags accounted for (July + 3 months), so July enrollment can be tested at every lag. Lags after July 2023 are flagged as end-of-window, and the rows are marked edge and end. The end of the analysis window is set by LIHEAP, and the final window is determined by the inner join with the outcome data
**<p>Mid Series Gaps</p>**
- There are missing lags because a zip code had no LIHEAP record are flagged as zip_gap. These are not zero-filled because a missing value in the lag column does not confirm zero enrollment.
**<p>Records dropped</p>**
  I removed only zip months with no values at all when building the lag table. All 852 rows have at least one usable lag are were retained in the final dataset.

**<p>Consistent across zip codes</p>**
Start and end bounds are dataset-wide (January 2023-July 2023), not per zip, so the same rules apply identically to all 121 zip codes 

**<p>How Incomplete rows are used</p>**
- single lag analysis: rows where the lag's status is present 
- joint analyses (all three lags): rows were Complete_Lags == True
- missing lags are never imputed in place 

  ### Flags Created
| Column | Values | Meaning |
|---|---|---|
| `lag1_status`, `lag2_status`, `lag3_status` | present / before_start / after_end / zip_gap | Why each lag is present or missing |
| `Complete_Lags` | True / False | All three lags present |
| `Edge` | start / end / core | Row affected by the start or end of the series |

### Results
| Edge | Incomplete | Complete |
|---|---|---|
| core | 168 | 346 |
| end | 165 | 0 |
| start | 173 | 0 |

- 852 zip-month rows total; 346 have complete lags, all within the core window
- 168 core rows are incomplete due to zip-level gaps, not series edges
- 77 of 121 zip codes (64%) contribute at least one complete-lag row; the
  remaining 44 have sporadic LIHEAP records and contribute none
- Contributing zips average 4–5 complete months (maximum possible: 5, April–August 2023)

**<p>Limitations</p>**
- Complete-lag results reflect higher-activity zip codes, since low-enrollment
  zips report too sporadically to form complete lag sequences.
- Many zip-month counts are very small (1–2 enrollments), so lag relationships
  should be assessed by pooling across zip codes rather than within individual zips.
- Counts above reflect the LIHEAP lag table only and will be updated after merging
  with food assistance outcomes.